# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question
**Decision:** Which visible pages, pages getting real search impressions, should a content editor review first because they're under capturing clicks relative to their position?

**Who acts, and how:** A content editor or SEO reviewer opens the flagged pages and decides whether to rewrite the title, meta description, or snippet, whether to escalate to engineering if tracking looks broken, or whether to schedule a content refresh.

**Why not a simple threshold:** Expected CTR varies roughly 6.4x by position tier (0.355 at page_1 down to 0.055 at deep), so a single fixed cutoff like "flag anything under 1%" would misclassify most pages. A page has to be compared only to others in its own position tier, with enough impressions to trust the comparison.

**Cost of a wrong call:** a false positive costs an editor's review time on a page that didn't need it. A false negative lets a real under capturing page keep losing clicks unnoticed.

**One paragraph frame:** for a content editor deciding which pages to review first, this project builds a ranked queue from FlyRank's starter search dataset, scoring pages by how far their CTR falls below their position tier's expected CTR, adjusted for volume, and separated by likely cause (a genuine CTR gap, a tracking anomaly, or page one content grown stale). The result claims only observed, directional, decision support patterns, never proof that a specific fix will work, and never a claim about how Google's algorithm itself behaves.



## 2. Data

**Release:** the FlyRank ML Internship starter dataset, `data/raw/content_refresh_anonymized.csv`, one row per content item (page), 30,000 rows across 44 columns, covering 32 pseudonymized clients. This is the sanctioned dataset for the CTR/Engagement Opportunity Scoring lane, the larger ~79 million row warehouse release was available but not required, and the starter slice already gave enough volume (22,006 to 30,000 pages depending on the trust floor applied) to build and validate a ranked queue.

**Date window:** all metrics are aggregated over a trailing 90-day window ending at the slice's export snapshot. No absolute calendar date is published for this slice specifically, unlike the full warehouse release, which is a fixed export dated 2026-07-03, this starter slice is a relative trailing window, not a dated point in time.

**What was excluded, and why:**

- **No client names, domains, URLs, raw queries, or credentials anywhere in the data.** `content_id` and `client_id` are pseudonymous identifiers (`content_` + 12 hex characters, `client_` + 10 hex characters), used only for grouping and joins, never as model features or shown in any output.

- **Rows below the trust floor were excluded from ranking, not deleted from analysis.** Pages with fewer than 500 impressions over the 90 day window don't have a reliable enough CTR to compare against a tier average, so they're visible in the underlying data but never enter the ranked queue.
- **`trend_direction` and `trend_pct` were excluded as features everywhere in this project,** per this dataset's own documented leakage warning, since the label is derived directly from trend_direction, using it as an input would be circular.

- **The full ~79 million row warehouse release was not used.** It's available and gated behind a free Hugging Face request, but the starter slice already supports this lane by design, using it kept the project's scope and runtime manageable across an 8 week track without sacrificing a valid result.

## 3. Methodology

**Label definition:** `ctr_gap = ctr - tier_avg_ctr`, where `tier_avg_ctr` is the mean CTR within each `position_tier`, computed only on pages with `impressions_90d >= 100` (the trust floor). This is a proxy label, not an outcome FlyRank measured, there is no ground truth "this page underperforms" flag in the data.

**Baseline (ML-07):** a rule, not a model. Flags a page when its CTR sits below its own tier's average, with volume as a secondary weight. By construction its R² on `ctr_gap` is 0, it uses only the columns the gap is built from, so it never "predicts" anything, it measures the gap directly.

**Assumptions:** a page's expected CTR depends on its position tier, not a single global average, confirmed by the roughly 6.4x spread between tiers. Pages need at least 100 impressions in the 90 day window to trust their CTR at all. All model features had to be knowable independent of the label's own ingredients.

**Features tested (Week 5):** `content_type`, `main_intent`, freshness_tier, age_tier, search_volume, competition, cpc, word_count, all confirmed independent of the label. Deliberately excluded: ctr, position_tier, tier_avg_ctr, avg_position (the label's direct ingredients), and impressions_90d (used only to define the trustworthy population, never as a feature).

**Validation design:** GroupShuffleSplit on client_id, 80/20, confirmed 0 shared clients between train and test. Pages from the same client share editorial and keyword patterns, a random row split would let the model see a client's quirks in both train and test and get credit for memorizing rather than generalizing.

**Leakage checks, three separate ones across the project:**

Week 3, an early sanity demonstration on real warehouse data: deliberately added ctr as a feature predicting ctr_gap. Honest model (position + impressions only): R² = 0.002. Leaky model (with ctr added): R² = 1.000, a perfect, meaningless score, since ctr_gap is algebraically just ctr minus a constant.
Week 6, naive vs. grouped split, same features and models as Week 5: a naive random row split gave misleadingly positive scores (Linear R² = 0.0431, Random Forest R² = 0.0917), with 28 of 30 clients appearing in both train and test. The honest, client grouped split (Week 5's actual design) revealed the real result: negative for both (Linear −0.5398, Random Forest −1.5655). Reporting the naive number would have looked like a working model; it was a leakage driven illusion.
Week 6, harness verification: deliberately reintroduced ctr into the honest, grouped-split pipeline. R² jumped from −1.5655 to 0.9204, confirming the test harness correctly rewards a real leak when one exists, so the negative results elsewhere were a genuine finding, not a broken pipeline silently returning garbage.

Root cause of the Random Forest's failure, found by ablation (Week 5): one training-only client, client_d4735e3a26 (61 rows), had a mean ctr_gap of 2.593, wildly outside every other client's range (next highest 0.841). Retraining without that client moved R² from −1.5655 to −0.0702, nearly matching the trivial DummyRegressor baseline of −0.0143. One noisy client, not a real content signal, was driving most of the model's damage; permutation importance corroborated this, every feature scored zero or negative importance, with content_type and age_tier (the categorical splits that client would pull hardest on) most negative.

**Conclusion carried into the rest of this project:** content and keyword metadata do not meaningfully explain the CTR gap in this dataset. The Week 4 rule was kept as the production method, not as a fallback, it was validated as the better choice. From ML-10 onward, that rule was refined into three archetypes (a genuine CTR gap, a likely tracking anomaly, and stale page-one content), each independently validated against the underlying data (population-preserving checks, a staleness fix that dropped a flawed archetype from 2,047 to 13 rows, a volume-dampened ranking that changed 7 of the old top 10).

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
